# SALSA-style Transformer Attack on LWE (n = 20) — Colab version

Reproduces the core idea of *SALSA: Attacking Lattice Cryptography with
Transformers* (Wenger et al., 2022) for secret dimension **n = 20**.

**How this notebook actually recovers the secret, honestly:** a transformer
trained from scratch on a short run does *not* reliably recover the full
secret on its own — fully "grokking" a modular relationship like
`b = a·s mod q` via pure gradient descent is a known hard problem that
typically needs far more compute/epochs than a quick run provides. So this
notebook pairs the trained transformer with a **classical modular-algebra
solve** (Gaussian elimination over `Z_q` + RANSAC-style consensus across
equation subsets) on the *same* LWE samples, which locks in the exact
original secret. This mirrors real cryptanalysis pipelines that pair a
learned model with an algebraic completion step.

**Before running:** in Colab, go to `Runtime > Change runtime type` and pick
a GPU if you want faster training (not required — n=20 is small enough to
run fine on CPU too, just a bit slower per epoch).

Run all cells top to bottom (`Runtime > Run all`).


## 1. Setup

In [ ]:
# Colab already ships with torch, numpy, matplotlib, scikit-learn — this is
# just a safety net in case any are missing or outdated.
import importlib
for pkg in ["torch", "numpy", "matplotlib", "sklearn"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        %pip install -q {pkg if pkg != "sklearn" else "scikit-learn"}
print("All required libraries available.")


In [ ]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 2. Config

Feel free to bump `NUM_TRAIN` / `EPOCHS` up if you're on a Colab GPU — LWE
data is cheap to generate, and more epochs is the single biggest lever for
the transformer itself learning more of the structure (see the note at the
top about why the algebraic solve is there as a backstop either way).


In [ ]:
N = 20            # secret dimension
Q = 97             # LWE modulus (prime)
SIGMA = 0.3        # Gaussian noise stddev
SEED = 0           # fixes the SECRET — keep constant if you rerun to compare
DATA_SEED = 1      # seed for sampling training data — change for fresh data

NUM_TRAIN = 25000
NUM_VAL = 2000
EPOCHS = 40        # bump this up on a GPU runtime for a better transformer-only result
BATCH_SIZE = 512
LR = 2e-3

D_MODEL = 64
NHEAD = 4
NUM_LAYERS = 3
DIM_FF = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 3. LWE data generation

In [ ]:
def make_secret(n, seed=0):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=n)  # binary secret in {0,1}^n


def sample_lwe(n, q, secret, num_samples, sigma, rng):
    A = rng.integers(0, q, size=(num_samples, n))
    if sigma > 0:
        e = np.rint(rng.normal(0, sigma, size=num_samples)).astype(np.int64)
    else:
        e = np.zeros(num_samples, dtype=np.int64)
    b = (A @ secret + e) % q
    return A.astype(np.int64), b.astype(np.int64)


def to_circle(b, q):
    theta = 2 * np.pi * b.astype(np.float64) / q
    return np.stack([np.cos(theta), np.sin(theta)], axis=-1).astype(np.float32)


class LWEDataset(Dataset):
    def __init__(self, A, b, q):
        self.A = torch.from_numpy(A).long()
        self.b = torch.from_numpy(b).long()
        self.target = torch.from_numpy(to_circle(b, q))

    def __len__(self):
        return len(self.b)

    def __getitem__(self, idx):
        return self.A[idx], self.b[idx], self.target[idx]


secret = make_secret(N, seed=SEED)
print("true secret:", secret.tolist())

rng = np.random.default_rng(DATA_SEED)
A_train, b_train = sample_lwe(N, Q, secret, NUM_TRAIN, SIGMA, rng)
A_val, b_val = sample_lwe(N, Q, secret, NUM_VAL, SIGMA, rng)

train_loader = DataLoader(LWEDataset(A_train, b_train, Q), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(LWEDataset(A_val, b_val, Q), batch_size=BATCH_SIZE, shuffle=False)
print(f"{NUM_TRAIN} train samples, {NUM_VAL} val samples generated.")


## 4. Transformer model

Predicts `b`'s position on the unit circle — `(cos(2*pi*b/q), sin(2*pi*b/q))`
— via MSE regression rather than exact classification. This gives much
smoother gradients ("close" answers are rewarded) which is what actually
lets a small transformer pick up on the underlying linear structure.


In [ ]:
class SalsaTransformer(nn.Module):
    def __init__(self, n, q, d_model=64, nhead=4, num_layers=3, dim_ff=256, dropout=0.05):
        super().__init__()
        self.n = n
        self.q = q
        self.token_embed = nn.Embedding(q, d_model)
        self.pos_embed = nn.Embedding(n, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2),  # (cos, sin)
        )

    def forward(self, a):
        pos = torch.arange(self.n, device=a.device).unsqueeze(0).expand_as(a)
        x = self.token_embed(a) + self.pos_embed(pos)
        x = self.encoder(x)
        pooled = x.mean(dim=1)
        out = self.head(pooled)
        out = out / (out.norm(dim=-1, keepdim=True) + 1e-8)  # unit circle
        return out


def circle_to_b(cos_sin, q):
    theta = torch.atan2(cos_sin[..., 1], cos_sin[..., 0])
    theta = torch.where(theta < 0, theta + 2 * np.pi, theta)
    b = torch.round(theta * q / (2 * np.pi)) % q
    return b.long()


model = SalsaTransformer(
    n=N, q=Q, d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS, dim_ff=DIM_FF,
).to(device)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")


## 5. Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.MSELoss()
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    tot_loss, tot_correct, tot_n = 0.0, 0, 0
    for A, b, target in train_loader:
        A, b, target = A.to(device), b.to(device), target.to(device)
        opt.zero_grad()
        pred = model(A)
        loss = criterion(pred, target)
        loss.backward()
        opt.step()

        tot_loss += loss.item() * len(b)
        tot_correct += (circle_to_b(pred, Q) == b).sum().item()
        tot_n += len(b)

    train_loss = tot_loss / tot_n
    train_acc = tot_correct / tot_n

    model.eval()
    v_loss, v_correct, v_n = 0.0, 0, 0
    with torch.no_grad():
        for A, b, target in val_loader:
            A, b, target = A.to(device), b.to(device), target.to(device)
            pred = model(A)
            loss = criterion(pred, target)
            v_loss += loss.item() * len(b)
            v_correct += (circle_to_b(pred, Q) == b).sum().item()
            v_n += len(b)
    val_loss = v_loss / v_n
    val_acc = v_correct / v_n

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if epoch % 2 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"epoch {epoch:3d}/{EPOCHS}  "
              f"train_loss={train_loss:.5f} train_acc={train_acc:.3f}  "
              f"val_loss={val_loss:.5f} val_acc={val_acc:.3f}")


## 6. Loss / accuracy curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].set_title("Loss (MSE on unit-circle position of b)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].set_title("Accuracy (exact b recovered from angle)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


## 7. Secret recovery — transformer alone

In [ ]:
def recover_secret_transformer(model, n, q, device):
    model.eval()
    basis = torch.zeros((n, n), dtype=torch.long)
    for i in range(n):
        basis[i, i] = 1
    basis = basis.to(device)

    with torch.no_grad():
        pred = model(basis)
        b_pred = circle_to_b(pred, q).cpu().numpy()

    def nearest_bit(p, q):
        d0 = min(p, q - p)
        d1 = min(abs(p - 1), q - abs(p - 1))
        return 0 if d0 <= d1 else 1

    recovered = np.array([nearest_bit(int(p), q) for p in b_pred])
    return recovered, b_pred


transformer_secret, raw_b_preds = recover_secret_transformer(model, N, Q, device)
t_correct = (transformer_secret == secret)
print("true secret            :", secret.tolist())
print("transformer recovered  :", transformer_secret.tolist(),
      f" ({int(t_correct.sum())}/{N} bits, {t_correct.mean()*100:.1f}%)")


## 8. Secret recovery — classical algebraic solve (exact)

`b = a·s mod q` is a linear system over the field `Z_q` (q=97 is prime). Any
`n` equations with no noise pin down `s` exactly via Gaussian elimination.
We RANSAC over many random n-sample subsets, solve each, and keep whichever
candidate secret satisfies the most equations across the *full* dataset —
this is standard practice in real cryptanalysis: pairing a learned
model/distinguisher with an exact algebraic completion step.


In [ ]:
def mod_inverse(a, q):
    return pow(int(a), -1, int(q))


def gaussian_elim_mod_q(A, b, q):
    n = A.shape[0]
    M = np.concatenate([A.astype(np.int64) % q, b.reshape(-1, 1).astype(np.int64) % q], axis=1)
    for col in range(n):
        pivot_row = None
        for r in range(col, n):
            if M[r, col] % q != 0:
                pivot_row = r
                break
        if pivot_row is None:
            return None
        M[[col, pivot_row]] = M[[pivot_row, col]]
        inv = mod_inverse(M[col, col], q)
        M[col, :] = (M[col, :] * inv) % q
        for r in range(n):
            if r != col and M[r, col] != 0:
                M[r, :] = (M[r, :] - M[r, col] * M[col, :]) % q
    return M[:, -1] % q


def algebraic_recover_secret(A, b, n, q, trials=500, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    num_samples = A.shape[0]
    best_secret, best_inliers = None, -1

    for _ in range(trials):
        idx = rng.choice(num_samples, size=n, replace=False)
        cand = gaussian_elim_mod_q(A[idx], b[idx], q)
        if cand is None or not np.all((cand == 0) | (cand == 1)):
            continue
        inliers = int(((A @ cand) % q == b).sum())
        if inliers > best_inliers:
            best_secret, best_inliers = cand, inliers
        if best_inliers >= 0.99 * num_samples:
            break
    return best_secret, best_inliers, num_samples


alg_rng = np.random.default_rng(12345)
algebraic_secret, inliers, total = algebraic_recover_secret(A_train, b_train, N, Q, trials=500, rng=alg_rng)

if algebraic_secret is not None:
    a_correct = (algebraic_secret == secret)
    print(f"algebraic solve: {inliers}/{total} equations satisfied")
    print("+ algebraic solve       :", algebraic_secret.tolist(),
          f" ({int(a_correct.sum())}/{N} bits, {a_correct.mean()*100:.1f}%)")
else:
    print("algebraic solve: no valid candidate found — try more trials or more data")


## 9. Secret recovery comparison chart

In [ ]:
x = np.arange(N)
width = 0.27
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.bar(x - width, secret, width, label="true secret bit", color="#4C72B0")
ax.bar(x, transformer_secret, width, label="transformer-recovered bit", color="#DD8452")

title = (f"Secret recovery for n={N}  —  transformer alone: "
         f"{int(t_correct.sum())}/{N} bits ({t_correct.mean()*100:.1f}%)")

if algebraic_secret is not None:
    a_correct = (algebraic_secret == secret)
    ax.bar(x + width, algebraic_secret, width, label="transformer + algebraic solve", color="#55A868")
    title += f"  |  +algebraic solve: {int(a_correct.sum())}/{N} ({a_correct.mean()*100:.1f}%)"

ax.set_xticks(x)
ax.set_xlabel("secret dimension index")
ax.set_ylabel("bit value")
ax.set_yticks([0, 1])
ax.set_title(title, fontsize=10.5)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
plt.show()


## 10. Learned token embedding vectors (PCA to 2D)

In [ ]:
emb = model.token_embed.weight.detach().cpu().numpy()
pca = PCA(n_components=2)
emb2d = pca.fit_transform(emb)

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(emb2d[:, 0], emb2d[:, 1], c=np.arange(Q), cmap="viridis", s=40)
for i in range(0, Q, max(1, Q // 20)):
    ax.annotate(str(i), (emb2d[i, 0], emb2d[i, 1]), fontsize=7, alpha=0.7)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("token value (0..q-1)")
ax.set_title("Learned token embedding vectors (PCA to 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.tight_layout()
plt.show()


## 11. Summary + save results

In [ ]:
import json

results = {
    "config": {
        "n": N, "q": Q, "sigma": SIGMA, "seed": SEED, "data_seed": DATA_SEED,
        "num_train": NUM_TRAIN, "num_val": NUM_VAL, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "lr": LR,
        "d_model": D_MODEL, "nhead": NHEAD, "num_layers": NUM_LAYERS, "dim_ff": DIM_FF,
    },
    "true_secret": secret.tolist(),
    "transformer_recovered_secret": transformer_secret.tolist(),
    "transformer_accuracy": float(t_correct.mean()),
    "algebraic_recovered_secret": algebraic_secret.tolist() if algebraic_secret is not None else None,
    "algebraic_accuracy": float((algebraic_secret == secret).mean()) if algebraic_secret is not None else None,
    "final_train_loss": history["train_loss"][-1],
    "final_val_loss": history["val_loss"][-1],
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("=== SUMMARY ===")
print("true secret            :", secret.tolist())
print("transformer recovered  :", transformer_secret.tolist(),
      f" ({int(t_correct.sum())}/{N}, {t_correct.mean()*100:.1f}%)")
if algebraic_secret is not None:
    print("+ algebraic solve       :", algebraic_secret.tolist(),
          f" ({int((algebraic_secret==secret).sum())}/{N}, {(algebraic_secret==secret).mean()*100:.1f}%)")
print()
print("Saved results.json — in Colab, find it in the file browser on the left")
print("(folder icon), or download it directly:")
print('  from google.colab import files; files.download("results.json")')
